# Data Understanding & Preprocessing

## Objective

To empirically understand the structure, quality, consistency,
relationships, and statistical characteristics of the project datasets
before data cleaning, preprocessing, feature engineering, and modeling.

In [224]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [225]:
raw_data_dir=Path("../data/raw/")
print(list(raw_data_dir.iterdir()))

[WindowsPath('../data/raw/bank_nifty_daily_raw.csv'), WindowsPath('../data/raw/constituent_change_events.csv'), WindowsPath('../data/raw/constituent_regimes.csv'), WindowsPath('../data/raw/constituent_regime_members.csv'), WindowsPath('../data/raw/constituent_stocks_raw.csv'), WindowsPath('../data/raw/constituent_weights_raw.csv'), WindowsPath('../data/raw/daily_weight_reconstruction.csv'), WindowsPath('../data/raw/earnings_raw.csv'), WindowsPath('../data/raw/index_weightage'), WindowsPath('../data/raw/weight_reconstruction_validation.csv')]


In [353]:
bank_nifty = pd.read_csv(raw_data_dir / "bank_nifty_daily_raw.csv")
constituent_change=pd.read_csv(raw_data_dir/"constituent_change_events.csv")
regimes= pd.read_csv(raw_data_dir / "constituent_regimes.csv")
const_reg_mem=pd.read_csv(raw_data_dir/"constituent_regime_members.csv")
constituents =pd.read_csv(raw_data_dir / "constituent_stocks_raw.csv")
con_mon_wts= pd.read_csv(raw_data_dir / "constituent_weights_raw.csv")
daily_weights= pd.read_csv(raw_data_dir / "daily_weight_reconstruction.csv")
earnings= pd.read_csv(raw_data_dir / "earnings_raw.csv")
wt_val= pd.read_csv(raw_data_dir/ "weight_reconstruction_validation.csv")

Understanding- Bank_Nifty_daily

In [227]:
# structural 
display(bank_nifty.shape)
display(bank_nifty.dtypes)
display(bank_nifty.head())
display(bank_nifty.info())
#missingness
display(bank_nifty.isna().sum())
display(bank_nifty.isna().mean()*100)
#duplicate records
display(bank_nifty.duplicated().sum())
display(bank_nifty['date'].duplicated().sum())

(907, 5)

date      object
open     float64
high     float64
low      float64
close    float64
dtype: object

,date,open,high,low,close
0,01-01-2024,48203.45,48450.00,48044.15,48234.30
1,01-01-2025,50841.90,51321.95,50485.05,51060.60
2,01-01-2026,59674.80,59743.20,59569.65,59711.55
3,01-02-2023,41115.00,42015.65,39490.50,40513.00
4,01-02-2024,46164.90,46306.90,45668.35,46188.65


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 907 entries, 0 to 906
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    907 non-null    object 
 1   open    907 non-null    float64
 2   high    907 non-null    float64
 3   low     907 non-null    float64
 4   close   907 non-null    float64
dtypes: float64(4), object(1)
memory usage: 35.6+ KB


None

date     0
open     0
high     0
low      0
close    0
dtype: int64

date     0.0
open     0.0
high     0.0
low      0.0
close    0.0
dtype: float64

np.int64(0)

np.int64(0)

In [228]:
#date understanding
bank_nifty["date_parsed"] = pd.to_datetime(
    bank_nifty["date"],
    format="%d-%m-%Y",
    errors="coerce"
) 
display(bank_nifty["date_parsed"].min())
display(bank_nifty["date_parsed"].max())
display(bank_nifty["date_parsed"].isna().sum())
display(bank_nifty["date_parsed"].is_monotonic_increasing)

Timestamp('2023-01-02 00:00:00')

Timestamp('2026-08-31 00:00:00')

np.int64(0)

False

In [229]:
#OHLC logical validity
display((
    (bank_nifty["high"] < bank_nifty["open"]) |
    (bank_nifty["high"] < bank_nifty["close"]) |
    (bank_nifty["high"] < bank_nifty["low"])
).sum())

display((
    (bank_nifty["low"] > bank_nifty["open"]) |
    (bank_nifty["low"] > bank_nifty["close"]) |
    (bank_nifty["low"] > bank_nifty["high"])
).sum())

#statistical understanding
bank_nifty[["open", "high", "low", "close"]].describe()

np.int64(0)

np.int64(0)

,open,high,low,close
count,907.000000,907.000000,907.000000,907.000000
mean,50752.645480,51037.013892,50446.675028,50748.548015
std,5680.763065,5708.918993,5681.690393,5699.930091
min,39061.700000,39381.400000,38613.150000,39051.500000
25%,45747.400000,46079.725000,45431.850000,45695.975000
50%,51084.950000,51420.150000,50795.900000,51143.850000
75%,55603.800000,55904.675000,55293.525000,55611.325000
max,61660.850000,61764.850000,61100.900000,61550.800000


understanding- constituents_change_events


In [230]:
display(constituent_change.shape)
display(constituent_change.dtypes)
display(constituent_change.head())

(4, 6)

effective_date     object
event_type         object
symbol             object
previous_status    object
new_status         object
source             object
dtype: object

,effective_date,event_type,symbol,previous_status,new_status,source
0,2024-09-30,EXCLUSION,BANDHANBNK,ACTIVE,INACTIVE,NSE Indices
1,2024-09-30,INCLUSION,CANBK,INACTIVE,ACTIVE,NSE Indices
2,2025-12-31,INCLUSION,UNIONBANK,INACTIVE,ACTIVE,NSE Indices
3,2025-12-31,INCLUSION,YESBANK,INACTIVE,ACTIVE,NSE Indices


In [231]:
display(constituent_change['effective_date'].unique())
constituent_change.groupby('new_status')['symbol'].unique()

array(['2024-09-30', '2025-12-31'], dtype=object)

new_status
ACTIVE      [CANBK, UNIONBANK, YESBANK]
INACTIVE                   [BANDHANBNK]
Name: symbol, dtype: object

understanding -constituent_regime_members

In [232]:
display(const_reg_mem.shape)
display(const_reg_mem.dtypes)
display(const_reg_mem.head())


(38, 3)

regime_id              object
start_snapshot_date    object
symbol                 object
dtype: object

,regime_id,start_snapshot_date,symbol
0,R1,2023-01-31,AUBANK
1,R1,2023-01-31,AXISBANK
2,R1,2023-01-31,BANDHANBNK
3,R1,2023-01-31,BANKBARODA
4,R1,2023-01-31,FEDERALBNK


In [233]:
#checking constituent consistency
display(const_reg_mem.groupby('regime_id')['symbol'].count())
regime_sets = (
    const_reg_mem.groupby('regime_id')['symbol']
    .apply(set)
)
inc=regime_sets['R2'] - regime_sets['R1'],regime_sets['R3'] - regime_sets['R2']
(set().union(*inc))

regime_id
R1    12
R2    12
R3    14
Name: symbol, dtype: int64

{'CANBK', 'UNIONBANK', 'YESBANK'}

understanding- consttuent_regimes

In [234]:
const_reg = pd.read_csv(raw_data_dir/"constituent_regimes.csv")

display(const_reg.shape)
display(const_reg.dtypes)
display(const_reg.head())

(3, 4)

regime_id              object
start_snapshot_date    object
end_snapshot_date      object
constituent_count       int64
dtype: object

,regime_id,start_snapshot_date,end_snapshot_date,constituent_count
0,R1,2023-01-31,2024-08-30,12
1,R2,2024-09-30,2025-11-28,12
2,R3,2025-12-31,2026-08-31,14


In [235]:
#validating constituent count in const_reg_mem against const_regime
const_reg_mem['regime_id'].value_counts(ascending=True) == const_reg.set_index('regime_id')['constituent_count']


regime_id
R1    True
R2    True
R3    True
dtype: bool

In [237]:
#checking validation of constituent_regime_members again
display(const_reg_mem.groupby('regime_id')['start_snapshot_date'].unique())

# checking validation of constituent_change_events again
constituent_change

regime_id
R1    [2023-01-31]
R2    [2024-09-30]
R3    [2025-12-31]
Name: start_snapshot_date, dtype: object

,effective_date,event_type,symbol,previous_status,new_status,source
0,2024-09-30,EXCLUSION,BANDHANBNK,ACTIVE,INACTIVE,NSE Indices
1,2024-09-30,INCLUSION,CANBK,INACTIVE,ACTIVE,NSE Indices
2,2025-12-31,INCLUSION,UNIONBANK,INACTIVE,ACTIVE,NSE Indices
3,2025-12-31,INCLUSION,YESBANK,INACTIVE,ACTIVE,NSE Indices


understanding- constituent_stocks_raw

In [238]:
display(constituents.shape)
display(constituents.dtypes)
display(constituents.head())
display(constituents.info())
#missingness
display(constituents.isna().sum())
#duplicate records
display(constituents.duplicated().sum())
display(constituents[['date','symbol']].duplicated().sum())

(13620, 7)

date       object
symbol     object
open      float64
high      float64
low       float64
close     float64
volume      int64
dtype: object

,date,symbol,open,high,low,close,volume
0,2023-01-02 00:00:00,AUBANK,656.90,661.25,654.0,657.00,1586054
1,2023-01-02 00:00:00,AXISBANK,932.25,945.00,931.8,941.60,3498198
2,2023-01-02 00:00:00,BANDHANBNK,236.30,242.30,235.5,240.15,7625641
3,2023-01-02 00:00:00,BANKBARODA,186.60,187.75,185.1,186.25,14679657
4,2023-01-02 00:00:00,CANBK,335.85,340.40,332.5,336.15,13632732


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13620 entries, 0 to 13619
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    13620 non-null  object 
 1   symbol  13620 non-null  object 
 2   open    13620 non-null  float64
 3   high    13620 non-null  float64
 4   low     13620 non-null  float64
 5   close   13620 non-null  float64
 6   volume  13620 non-null  int64  
dtypes: float64(4), int64(1), object(2)
memory usage: 745.0+ KB


None

date      0
symbol    0
open      0
high      0
low       0
close     0
volume    0
dtype: int64

date      0.0
symbol    0.0
open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
dtype: float64

np.int64(0)

np.int64(0)

In [239]:
constituents["date_parsed"] = pd.to_datetime(
    constituents["date"],
    errors="coerce"
) 
display(constituents["date_parsed"].min())
display(constituents["date_parsed"].max())
display(constituents["date_parsed"].isna().sum())
display(constituents["date_parsed"].is_monotonic_increasing)

Timestamp('2023-01-02 00:00:00')

Timestamp('2026-08-31 00:00:00')

np.int64(0)

True

In [240]:
#OHLC Logical Validity
display((
    (constituents["high"] < constituents["open"]) |
    (constituents["high"] < constituents["close"]) |
    (constituents["high"] < constituents["low"])
).sum())

display((
    (constituents["low"] > constituents["open"]) |
    (constituents["low"] > constituents["close"]) |
    (constituents["low"] > constituents["high"])
).sum())

np.int64(0)

np.int64(0)

In [241]:
display(len(constituents['symbol'].unique()),
constituents['symbol'].unique())

15

array(['AUBANK', 'AXISBANK', 'BANDHANBNK', 'BANKBARODA', 'CANBK',
       'FEDERALBNK', 'HDFCBANK', 'ICICIBANK', 'IDFCFIRSTB', 'INDUSINDBK',
       'KOTAKBANK', 'PNB', 'SBIN', 'UNIONBANK', 'YESBANK'], dtype=object)

In [242]:
# referential integrity check
pd.to_datetime(regimes['start_snapshot_date'])
pd.to_datetime(regimes['end_snapshot_date'])
pd.to_datetime(constituent_change['effective_date'])
regime_check=regimes[['regime_id','start_snapshot_date']].merge(constituent_change[["effective_date"]],left_on='start_snapshot_date',right_on='effective_date',how='left', indicator=True)
regime_check

,regime_id,start_snapshot_date,effective_date,_merge
0,R1,2023-01-31,NaN,left_only
1,R2,2024-09-30,2024-09-30,both
2,R2,2024-09-30,2024-09-30,both
3,R3,2025-12-31,2025-12-31,both
4,R3,2025-12-31,2025-12-31,both


In [249]:
#checking trading days count match
display(bank_nifty.shape[0]==constituents['date_parsed'].nunique())
display(constituents['date_parsed'].nunique())
display(bank_nifty['date'].nunique())
display(constituents['date_parsed'].min())
display(bank_nifty['date_parsed'].min())
display(constituents.shape[0] - constituents['date'].duplicated().sum())

False

908

907

Timestamp('2023-01-02 00:00:00')

Timestamp('2023-01-02 00:00:00')

np.int64(908)

In [245]:
constituents['date_parsed']=pd.to_datetime(constituents['date'],errors='coerce')
bank_nifty['date_parsed']=pd.to_datetime(bank_nifty['date'],errors='coerce')

In [246]:
#checking data completeness
regimes['start_snapshot_date']=pd.to_datetime(regimes['start_snapshot_date'])
regimes['end_snapshot_date']=pd.to_datetime(regimes['end_snapshot_date'])

constituents['regime_id'] = None

for _, regime in regimes.iterrows():
    mask = ((constituents['date_parsed']>=regime['start_snapshot_date']) & (constituents['date_parsed'] <= regime['end_snapshot_date']))
    constituents.loc[mask, "regime_id"] = regime['regime_id']

actual_counts = (
    constituents.groupby('regime_id')['symbol']
    .nunique()
    .reset_index(name="actual_count")
)

validation = regimes.merge(
    actual_counts,
    on="regime_id"
)

validation['match'] = (validation['constituent_count'] == validation['actual_count'])

validation

,regime_id,start_snapshot_date,end_snapshot_date,constituent_count,actual_count,match
0,R1,2023-01-31,2024-08-30,12,15,False
1,R2,2024-09-30,2025-11-28,12,15,False
2,R3,2025-12-31,2026-08-31,14,15,False


In [255]:
#understanding data spread
pd.pivot_table(constituents, index='symbol',values=['open','low','high','close','volume'],aggfunc=['mean','median','std'])

mean                                                       \
                  close         high          low         open        volume   
symbol                                                                         
AUBANK       744.485738   754.004570   734.648513   744.928910  2.639374e+06   
AXISBANK    1118.271641  1129.201762  1107.248844  1118.221476  8.648406e+06   
BANDHANBNK   192.386145   195.369053   189.754923   192.788921  1.091935e+07   
BANKBARODA   237.805055   240.915694   234.829570   237.986817  1.515898e+07   
CANBK        220.298458   223.372137   217.350231   220.541355  2.021540e+07   
FEDERALBNK   197.272753   199.425661   194.925374   197.336366  1.138538e+07   
HDFCBANK    1449.569824  1461.084031  1438.551211  1449.509031  2.040971e+07   
ICICIBANK   1190.724559  1200.295760  1180.633095  1190.253744  1.425473e+07   
IDFCFIRSTB    74.009471    74.911674    73.133458    74.065430  3.315351e+07   
INDUSINDBK  1129.687445  1143.455562  1116.038932  1130.881002  4.585228e+06   
KOTAKBANK   1628.859967  1643.056057  1614.524670  1628.907985  6.733094e+06   
PNB           98.193678    99.563128    96.901233    98.326377  3.752577e+07   
SBIN         789.053469   796.762004   781.434086   789.652423  1.469140e+07   
UNIONBANK    128.354581   130.429681   126.436013   128.532709  1.590860e+07   
YESBANK       20.533106    20.882137    20.276278    20.582963  1.504018e+08   

              median                                                   std  \
               close      high       low      open      volume       close   
symbol                                                                       
AUBANK       716.950   723.700   709.050   718.000   1945272.0  152.590317   
AXISBANK    1131.100  1141.375  1118.075  1130.750   7460706.5  137.612753   
BANDHANBNK   188.400   191.925   185.845   189.200   8277347.5   32.537964   
BANKBARODA   243.220   246.000   240.775   243.430  12768174.5   36.718501   
CANBK        134.810   136.780   133.295   134.970  16681563.0  148.564061   
FEDERALBNK   190.665   193.225   188.315   190.945   8852487.0   60.486602   
HDFCBANK    1603.425  1613.725  1589.975  1603.975  16357675.0  382.688931   
ICICIBANK   1241.575  1252.825  1230.050  1239.475  12484839.0  195.568126   
IDFCFIRSTB    73.770    74.640    73.000    73.765  25865177.5   10.289773   
INDUSINDBK  1055.200  1069.100  1035.575  1056.000   3173425.5  280.096130   
KOTAKBANK   1791.525  1809.750  1777.250  1795.000   4120737.5  578.959006   
PNB          105.220   106.470   104.110   105.225  27705986.5   25.064855   
SBIN         797.425   804.300   791.375   798.800  12935721.5  171.090396   
UNIONBANK    129.280   131.475   127.700   129.395  12220172.5   33.349575   
YESBANK       20.720    21.075    20.355    20.755  99307305.5    3.007061   

                                                              
                  high         low        open        volume  
symbol                                                        
AUBANK      153.584471  150.702195  151.924513  2.613739e+06  
AXISBANK    139.253307  136.223648  137.637896  4.987980e+06  
BANDHANBNK   32.994259   32.143960   32.712729  1.182469e+07  
BANKBARODA   37.143375   36.316378   36.701626  1.131027e+07  
CANBK       150.725379  146.411638  148.695237  1.833410e+07  
FEDERALBNK   60.960583   59.677135   60.219919  1.141416e+07  
HDFCBANK    385.043570  380.075849  382.796255  1.725141e+07  
ICICIBANK   196.705064  194.341341  195.130558  7.709862e+06  
IDFCFIRSTB   10.366717   10.243966   10.350278  3.385073e+07  
INDUSINDBK  281.755008  278.628288  281.228135  6.726110e+06  
KOTAKBANK   583.722145  574.190310  579.085887  7.555657e+06  
PNB          25.394332   24.774227   25.132170  3.165111e+07  
SBIN        172.764926  169.202751  170.936537  8.857548e+06  
UNIONBANK    33.782775   32.824799   33.310353  1.420333e+07  
YESBANK       3.111195    2.939475    3.036669  1.624277e+08

understanding- constituents_weights_raw


In [260]:
display(con_mon_wts.shape)
display(con_mon_wts.dtypes)
display(con_mon_wts.head())
display(con_mon_wts.info())
#missingness
display(con_mon_wts.isna().sum())
#duplicate records
display(con_mon_wts.duplicated().sum())
display(con_mon_wts[['date','symbol']].duplicated().sum())

(546, 10)

date                        object
symbol                      object
security_name               object
industry                    object
close_price                float64
index_mcap_rs_crores         int64
weightage_pct              float64
source_month                object
source_file                 object
weight_observation_type     object
dtype: object

,date,symbol,security_name,industry,close_price,index_mcap_rs_crores,weightage_pct,source_month,source_file,weight_observation_type
0,2023-01-31,AUBANK,AU Small Finance Bank Ltd.,Financial Services,618.90,54676,2.49,2023-01,NIFTY_Bank_Jan2023.pdf,MONTH_END
1,2023-01-31,AXISBANK,Axis Bank Ltd.,Financial Services,871.60,220054,10.01,2023-01,NIFTY_Bank_Jan2023.pdf,MONTH_END
2,2023-01-31,BANDHANBNK,Bandhan Bank Ltd.,Financial Services,244.20,34039,1.55,2023-01,NIFTY_Bank_Jan2023.pdf,MONTH_END
3,2023-01-31,BANKBARODA,Bank of Baroda,Financial Services,167.85,57532,2.62,2023-01,NIFTY_Bank_Jan2023.pdf,MONTH_END
4,2023-01-31,FEDERALBNK,Federal Bank Ltd.,Financial Services,134.45,52359,2.38,2023-01,NIFTY_Bank_Jan2023.pdf,MONTH_END


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546 entries, 0 to 545
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     546 non-null    object 
 1   symbol                   546 non-null    object 
 2   security_name            546 non-null    object 
 3   industry                 546 non-null    object 
 4   close_price              546 non-null    float64
 5   index_mcap_rs_crores     546 non-null    int64  
 6   weightage_pct            546 non-null    float64
 7   source_month             546 non-null    object 
 8   source_file              546 non-null    object 
 9   weight_observation_type  546 non-null    object 
dtypes: float64(2), int64(1), object(7)
memory usage: 42.8+ KB


None

date                       0
symbol                     0
security_name              0
industry                   0
close_price                0
index_mcap_rs_crores       0
weightage_pct              0
source_month               0
source_file                0
weight_observation_type    0
dtype: int64

np.int64(0)

np.int64(0)

In [288]:
# checking constituent percentage summation for each month 
con_mon_wts['date']=pd.to_datetime(con_mon_wts['date'])
con_mon_wts.groupby('date')['weightage_pct'].sum()


0      False
1      False
2      False
3      False
4      False
       ...  
541    False
542    False
543    False
544    False
545    False
Name: date, Length: 546, dtype: bool

In [293]:
#checking missing/extra months
actual_months= pd.Index(con_mon_wts['date'].dt.to_period("M").unique())
expected_months = pd.period_range(start='2023-01', end='2026-08',freq='M')
missing_months= expected_months.difference(actual_months)
extra_months= actual_months.difference(expected_months)
display(missing_months)
display(extra_months)

PeriodIndex([], dtype='period[M]')

PeriodIndex([], dtype='period[M]')

In [295]:
#checking weightage_pct logical validation
display(con_mon_wts['weightage_pct'].describe())
display(con_mon_wts[con_mon_wts['weightage_pct']==con_mon_wts['weightage_pct'].min()])

count    546.000000
mean       8.058736
std        7.925994
min        0.800000
25%        2.612500
50%        4.290000
75%        9.870000
max       29.700000
Name: weightage_pct, dtype: float64

,date,symbol,security_name,industry,close_price,index_mcap_rs_crores,weightage_pct,source_month,source_file,weight_observation_type
432,2025-12-31,UNIONBANK,Union Bank of India,Financial Services,153.76,32315,0.8,2025-12,NIFTY_Bank_Dec2025.pdf,MONTH_END


Understanding- daily_weight_reconstruction

In [298]:
display(daily_weights.shape)
display(daily_weights.dtypes)
display(daily_weights.head())
display(daily_weights.info())
#missingness
display(daily_weights.isna().sum())
#duplicate records
display(daily_weights.duplicated().sum())
display(daily_weights[['date','symbol']].duplicated().sum())

(10494, 4)

date              object
symbol            object
weight           float64
weight_source     object
dtype: object

,date,symbol,weight,weight_source
0,2023-01-31,AUBANK,2.49,OFFICIAL
1,2023-01-31,AXISBANK,10.01,OFFICIAL
2,2023-01-31,BANDHANBNK,1.55,OFFICIAL
3,2023-01-31,BANKBARODA,2.62,OFFICIAL
4,2023-01-31,FEDERALBNK,2.38,OFFICIAL


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10494 entries, 0 to 10493
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           10494 non-null  object 
 1   symbol         10494 non-null  object 
 2   weight         10494 non-null  float64
 3   weight_source  10494 non-null  object 
dtypes: float64(1), object(3)
memory usage: 328.1+ KB


None

date             0
symbol           0
weight           0
weight_source    0
dtype: int64

np.int64(0)

np.int64(0)

In [304]:
#checking missing/extra months
daily_weights['date']=pd.to_datetime(daily_weights['date'])
display(daily_weights['date'].isna().sum())
display(daily_weights['date'].max())
display(daily_weights['date'].min())
actual_months= pd.Index(daily_weights['date'].dt.to_period("M").unique())
expected_months = pd.period_range(start='2023-01', end='2026-08',freq='M')
missing_months= expected_months.difference(actual_months)
extra_months= actual_months.difference(expected_months)
display(missing_months)
display(extra_months)

np.int64(0)

Timestamp('2026-08-31 00:00:00')

Timestamp('2023-01-31 00:00:00')

PeriodIndex([], dtype='period[M]')

PeriodIndex([], dtype='period[M]')

In [314]:
# weight integrity check
display(daily_weights['weight'].describe())
display(daily_weights[daily_weights['weight']==daily_weights['weight'].max()])
display(daily_weights[daily_weights['date']=="2025-07-31"])
display(daily_weights[daily_weights['date']=="2025-08-01"])
constituents['date']=pd.to_datetime(constituents['date'])
display(constituents[constituents['date']=="2025-07-31"])
display(constituents[constituents['date']=="2025-08-01"])

count    10494.000000
mean         8.071285
std          8.024951
min          0.800000
25%          2.564588
50%          4.308447
75%          9.779413
max         45.175661
Name: weight, dtype: float64

,date,symbol,weight,weight_source
7217,2025-08-01,HDFCBANK,45.175661,RECONSTRUCTED


,date,symbol,weight,weight_source
7200,2025-07-31,AUBANK,2.76,OFFICIAL
7201,2025-07-31,AXISBANK,7.67,OFFICIAL
7202,2025-07-31,BANKBARODA,2.92,OFFICIAL
7203,2025-07-31,CANBK,2.40,OFFICIAL
7204,2025-07-31,FEDERALBNK,3.29,OFFICIAL
7205,2025-07-31,HDFCBANK,29.09,OFFICIAL
7206,2025-07-31,ICICIBANK,26.47,OFFICIAL
7207,2025-07-31,IDFCFIRSTB,3.01,OFFICIAL
7208,2025-07-31,INDUSINDBK,3.49,OFFICIAL
7209,2025-07-31,KOTAKBANK,7.83,OFFICIAL


,date,symbol,weight,weight_source
7212,2025-08-01,AUBANK,2.150141,RECONSTRUCTED
7213,2025-08-01,AXISBANK,5.935610,RECONSTRUCTED
7214,2025-08-01,BANKBARODA,2.247270,RECONSTRUCTED
7215,2025-08-01,CANBK,1.842966,RECONSTRUCTED
7216,2025-08-01,FEDERALBNK,2.481976,RECONSTRUCTED
7217,2025-08-01,HDFCBANK,45.175661,RECONSTRUCTED
7218,2025-08-01,ICICIBANK,20.479645,RECONSTRUCTED
7219,2025-08-01,IDFCFIRSTB,2.308360,RECONSTRUCTED
7220,2025-08-01,INDUSINDBK,2.666929,RECONSTRUCTED
7221,2025-08-01,KOTAKBANK,6.147648,RECONSTRUCTED


,date,symbol,open,high,low,close,volume
9600,2025-07-31,AUBANK,742.75,749.75,736.75,741.50,3573228
9601,2025-07-31,AXISBANK,1066.00,1081.30,1063.90,1068.40,7634413
9602,2025-07-31,BANDHANBNK,167.20,169.71,166.42,168.08,7725352
9603,2025-07-31,BANKBARODA,237.50,240.95,236.50,237.87,5770571
9604,2025-07-31,CANBK,107.05,108.36,106.10,107.25,48203474
9605,2025-07-31,FEDERALBNK,203.00,204.73,201.14,202.43,8617317
9606,2025-07-31,HDFCBANK,2014.60,2031.40,2008.00,2018.20,11091516
9607,2025-07-31,ICICIBANK,1469.90,1494.50,1464.00,1481.40,12406146
9608,2025-07-31,IDFCFIRSTB,68.50,69.62,68.23,68.76,22673335
9609,2025-07-31,INDUSINDBK,795.60,806.00,789.10,798.90,4480429


,date,symbol,open,high,low,close,volume
9615,2025-08-01,AUBANK,741.50,751.45,739.05,741.05,2251668
9616,2025-08-01,AXISBANK,1065.00,1070.90,1060.00,1062.40,5545447
9617,2025-08-01,BANDHANBNK,168.00,168.58,163.00,163.42,3579216
9618,2025-08-01,BANKBARODA,237.74,238.61,234.55,235.13,4770791
9619,2025-08-01,CANBK,107.25,107.40,105.75,105.89,13286442
9620,2025-08-01,FEDERALBNK,202.68,202.92,194.41,196.06,10948324
9621,2025-08-01,HDFCBANK,2014.00,2028.30,2009.30,2012.20,5658629
9622,2025-08-01,ICICIBANK,1480.40,1485.10,1469.80,1471.60,4416771
9623,2025-08-01,IDFCFIRSTB,68.90,68.95,67.46,67.58,13817940
9624,2025-08-01,INDUSINDBK,792.50,798.65,779.70,783.70,4739250


In [329]:
#grave issues in weight reconstruction methodology
display(daily_weights[(daily_weights['weight']>29) & (daily_weights['weight']<=daily_weights['weight'].max())])
display(daily_weights[daily_weights['date']=="2026-02-27"])

,date,symbol,weight,weight_source
17,2023-02-01,HDFCBANK,29.184456,RECONSTRUCTED
41,2023-02-03,HDFCBANK,29.037640,RECONSTRUCTED
53,2023-02-06,HDFCBANK,29.001635,RECONSTRUCTED
113,2023-02-13,HDFCBANK,29.182625,RECONSTRUCTED
125,2023-02-14,HDFCBANK,29.193875,RECONSTRUCTED
...,...,...,...,...
8263,2026-01-07,KOTAKBANK,33.753930,RECONSTRUCTED
8277,2026-01-08,KOTAKBANK,33.773317,RECONSTRUCTED
8291,2026-01-09,KOTAKBANK,33.864816,RECONSTRUCTED
8305,2026-01-12,KOTAKBANK,33.856081,RECONSTRUCTED


,date,symbol,weight,weight_source
8758,2026-02-27,AUBANK,3.96,OFFICIAL
8759,2026-02-27,AXISBANK,9.94,OFFICIAL
8760,2026-02-27,BANKBARODA,4.45,OFFICIAL
8761,2026-02-27,CANBK,3.90,OFFICIAL
8762,2026-02-27,FEDERALBNK,5.55,OFFICIAL
8763,2026-02-27,HDFCBANK,19.69,OFFICIAL
8764,2026-02-27,ICICIBANK,16.13,OFFICIAL
8765,2026-02-27,IDFCFIRSTB,3.60,OFFICIAL
8766,2026-02-27,INDUSINDBK,4.72,OFFICIAL
8767,2026-02-27,KOTAKBANK,9.16,OFFICIAL


In [347]:
kotak_filtr=daily_weights[daily_weights['symbol']=='KOTAKBANK']
display(kotak_filtr[kotak_filtr['weight']>10.6])

,date,symbol,weight,weight_source
969,2023-05-31,KOTAKBANK,10.690000,OFFICIAL
981,2023-06-01,KOTAKBANK,10.692844,RECONSTRUCTED
993,2023-06-02,KOTAKBANK,10.705111,RECONSTRUCTED
1017,2023-06-06,KOTAKBANK,10.763223,RECONSTRUCTED
1029,2023-06-07,KOTAKBANK,10.630138,RECONSTRUCTED
8207,2026-01-01,KOTAKBANK,34.722666,RECONSTRUCTED
8221,2026-01-02,KOTAKBANK,34.280112,RECONSTRUCTED
8235,2026-01-05,KOTAKBANK,34.263486,RECONSTRUCTED
8249,2026-01-06,KOTAKBANK,33.734937,RECONSTRUCTED
8263,2026-01-07,KOTAKBANK,33.753930,RECONSTRUCTED


In [352]:
daily_weights[daily_weights['symbol']=='HDFCBANK']

,date,symbol,weight,weight_source
5,2023-01-31,HDFCBANK,28.660000,OFFICIAL
17,2023-02-01,HDFCBANK,29.184456,RECONSTRUCTED
29,2023-02-02,HDFCBANK,28.640046,RECONSTRUCTED
41,2023-02-03,HDFCBANK,29.037640,RECONSTRUCTED
53,2023-02-06,HDFCBANK,29.001635,RECONSTRUCTED
...,...,...,...,...
10429,2026-08-25,HDFCBANK,17.617487,RECONSTRUCTED
10443,2026-08-26,HDFCBANK,17.528060,RECONSTRUCTED
10457,2026-08-27,HDFCBANK,17.219142,RECONSTRUCTED
10471,2026-08-28,HDFCBANK,17.448533,RECONSTRUCTED


In [350]:
daily_weights

,date,symbol,weight,weight_source
0,2023-01-31,AUBANK,2.49,OFFICIAL
1,2023-01-31,AXISBANK,10.01,OFFICIAL
2,2023-01-31,BANDHANBNK,1.55,OFFICIAL
3,2023-01-31,BANKBARODA,2.62,OFFICIAL
4,2023-01-31,FEDERALBNK,2.38,OFFICIAL
...,...,...,...,...
10489,2026-08-31,KOTAKBANK,9.88,OFFICIAL
10490,2026-08-31,PNB,3.37,OFFICIAL
10491,2026-08-31,SBIN,10.27,OFFICIAL
10492,2026-08-31,UNIONBANK,3.17,OFFICIAL


Understanding =earnings_raw

In [338]:
display(earnings.shape)
display(earnings.dtypes)
display(earnings.head())
display(earnings.info())
#missingness
display(earnings.isna().sum())
#duplicate records
display(earnings.duplicated().sum())
display(earnings[['date','symbol']].duplicated().sum())

(240, 10)

symbol              object
date                object
period_end          object
eps_actual          object
eps_forecast        object
revenue_actual      object
revenue_forecast    object
eps_surprise        object
revenue_surprise    object
reaction            object
dtype: object

,symbol,date,period_end,eps_actual,eps_forecast,revenue_actual,revenue_forecast,eps_surprise,revenue_surprise,reaction
0,AXISBANK,2026-10-15,09/2026,--,17.50,--,220.00B,--,--,--
1,AXISBANK,2026-07-18,06/2026,22.90,22.99,213.82B,216.43B,-0.39%,-1.21%,-5.46%
2,AXISBANK,2026-04-25,03/2026,22.80,22.80,204.80B,218.03B,0%,-6.07%,-3.05%
3,AXISBANK,2026-01-26,12/2025,20.90,18.77,205.12B,208.98B,+11.35%,-1.85%,+4.59%
4,AXISBANK,2025-10-15,09/2025,17.72,16.27,323.10B,199.35B,+8.91%,+62.08%,+1.66%


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   symbol            240 non-null    object
 1   date              240 non-null    object
 2   period_end        240 non-null    object
 3   eps_actual        240 non-null    object
 4   eps_forecast      240 non-null    object
 5   revenue_actual    240 non-null    object
 6   revenue_forecast  240 non-null    object
 7   eps_surprise      240 non-null    object
 8   revenue_surprise  240 non-null    object
 9   reaction          240 non-null    object
dtypes: object(10)
memory usage: 18.9+ KB


None

symbol              0
date                0
period_end          0
eps_actual          0
eps_forecast        0
revenue_actual      0
revenue_forecast    0
eps_surprise        0
revenue_surprise    0
reaction            0
dtype: int64

np.int64(0)

np.int64(0)

Understanding - weight_reconstruction_validation

In [355]:
display(wt_val.shape)
display(wt_val.dtypes)
display(wt_val.head())
display(wt_val.info())
#missingness
display(wt_val.isna().sum())
#duplicate records
display(wt_val.duplicated().sum())

(41, 11)

regime_id                    object
later_anchor                 object
earlier_anchor               object
constituent_count             int64
mae                         float64
rmse                        float64
max_abs_error               float64
correlation                 float64
reconstructed_weight_sum    float64
official_weight_sum         float64
status                       object
dtype: object

,regime_id,later_anchor,earlier_anchor,constituent_count,mae,rmse,max_abs_error,correlation,reconstructed_weight_sum,official_weight_sum,status
0,R1,2023-02-28,2023-01-31,12,0.003671,0.004454,0.008241,1.000000,100.0,100.01,validated
1,R1,2023-03-31,2023-02-28,12,0.330834,0.454100,0.962596,0.999401,100.0,100.01,validated
2,R1,2023-04-28,2023-03-31,12,0.003813,0.004352,0.007680,1.000000,100.0,100.00,validated
3,R1,2023-05-31,2023-04-28,12,0.003246,0.003812,0.005604,1.000000,100.0,100.00,validated
4,R1,2023-06-30,2023-05-31,12,0.396830,0.485467,0.959948,0.999533,100.0,100.03,validated


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   regime_id                 41 non-null     object 
 1   later_anchor              41 non-null     object 
 2   earlier_anchor            41 non-null     object 
 3   constituent_count         41 non-null     int64  
 4   mae                       41 non-null     float64
 5   rmse                      41 non-null     float64
 6   max_abs_error             41 non-null     float64
 7   correlation               41 non-null     float64
 8   reconstructed_weight_sum  41 non-null     float64
 9   official_weight_sum       41 non-null     float64
 10  status                    41 non-null     object 
dtypes: float64(6), int64(1), object(4)
memory usage: 3.7+ KB


None

regime_id                   0
later_anchor                0
earlier_anchor              0
constituent_count           0
mae                         0
rmse                        0
max_abs_error               0
correlation                 0
reconstructed_weight_sum    0
official_weight_sum         0
status                      0
dtype: int64

np.int64(0)

In [358]:
#range of validation check
wt_val['later_anchor']=pd.to_datetime(wt_val['later_anchor'])
wt_val['earlier_anchor']=pd.to_datetime(wt_val['earlier_anchor'])
display(wt_val['earlier_anchor'].min())
display(wt_val['later_anchor'].min())

display(wt_val['earlier_anchor'].max())
display(wt_val['later_anchor'].max())

Timestamp('2023-01-31 00:00:00')

Timestamp('2023-02-28 00:00:00')

Timestamp('2026-07-31 00:00:00')

Timestamp('2026-08-31 00:00:00')

In [361]:
actual_months = wt_val['later_anchor'].dt.to_period("M").unique()
expected_months = pd.period_range(start="2023-02",end="2026-08",freq="M")

missing_months = expected_months.difference(actual_months)

missing_months

PeriodIndex(['2024-09', '2025-12'], dtype='period[M]')

In [362]:
wt_val.describe()

,later_anchor,earlier_anchor,constituent_count,mae,rmse,max_abs_error,correlation,reconstructed_weight_sum,official_weight_sum
count,41,41,41.000000,41.000000,41.000000,41.000000,41.000000,4.100000e+01,41.000000
mean,2024-11-20 15:13:10.243902464,2024-10-21 07:01:27.804878080,12.390244,0.319893,0.540775,1.614474,0.986898,1.000000e+02,100.001463
min,2023-02-28 00:00:00,2023-01-31 00:00:00,12.000000,0.001680,0.002209,0.003605,0.576622,1.000000e+02,99.980000
25%,2023-12-29 00:00:00,2023-11-30 00:00:00,12.000000,0.003450,0.004003,0.007439,0.998066,1.000000e+02,100.000000
50%,2024-11-29 00:00:00,2024-10-31 00:00:00,12.000000,0.005305,0.005692,0.011503,0.999999,1.000000e+02,100.000000
75%,2025-09-30 00:00:00,2025-08-29 00:00:00,12.000000,0.396830,0.485467,1.120231,1.000000,1.000000e+02,100.010000
max,2026-08-31 00:00:00,2026-07-31 00:00:00,14.000000,3.712050,7.456577,25.362814,1.000000,1.000000e+02,100.030000
std,NaN,NaN,0.802435,0.707310,1.378335,4.582170,0.066598,1.507289e-14,0.009634


In [382]:
#checking missing months in daily_wts and constituent_stocks
daily_weights['date'] = pd.to_datetime(daily_weights['date'])

actual_months = daily_weights['date'].dt.to_period("M").drop_duplicates().sort_values()
expected_months = pd.period_range(
    start=daily_weights['date'].min().strftime('%Y-%m'),
    end = daily_weights['date'].max().strftime('%Y-%m'),
    freq = "M"
)

missing_months = expected_months.difference(actual_months)

display(missing_months)

display(daily_weights[daily_weights['date'] == "2025-11-28"])

constituents

PeriodIndex([], dtype='period[M]')

,date,symbol,weight,weight_source
8172,2025-11-28,AUBANK,3.19,OFFICIAL
8173,2025-11-28,AXISBANK,9.10,OFFICIAL
8174,2025-11-28,BANKBARODA,3.18,OFFICIAL
8175,2025-11-28,CANBK,3.04,OFFICIAL
8176,2025-11-28,FEDERALBNK,3.77,OFFICIAL
8177,2025-11-28,HDFCBANK,27.65,OFFICIAL
8178,2025-11-28,ICICIBANK,23.00,OFFICIAL
8179,2025-11-28,IDFCFIRSTB,2.99,OFFICIAL
8180,2025-11-28,INDUSINDBK,3.36,OFFICIAL
8181,2025-11-28,KOTAKBANK,8.75,OFFICIAL


,date,symbol,open,high,low,close,volume
0,2023-01-02 00:00:00,AUBANK,656.90,661.25,654.00,657.00,1586054
1,2023-01-02 00:00:00,AXISBANK,932.25,945.00,931.80,941.60,3498198
2,2023-01-02 00:00:00,BANDHANBNK,236.30,242.30,235.50,240.15,7625641
3,2023-01-02 00:00:00,BANKBARODA,186.60,187.75,185.10,186.25,14679657
4,2023-01-02 00:00:00,CANBK,335.85,340.40,332.50,336.15,13632732
...,...,...,...,...,...,...,...
13615,2026-08-31 00:00:00,KOTAKBANK,421.00,425.05,419.40,419.40,18992797
13616,2026-08-31 00:00:00,PNB,115.00,115.35,113.16,114.23,18423594
13617,2026-08-31 00:00:00,SBIN,1041.20,1060.00,1034.50,1060.00,5138732
13618,2026-08-31 00:00:00,UNIONBANK,184.68,188.63,181.80,188.63,9767401


In [388]:
#checking whether bank_nifty_daily or constituent_stocks_data have any O=H=L=C
stocks=pd.read_csv("../data/processed/constituent_stocks_data.csv")
bank_nifty=pd.read_csv("../data/processed/bank_nifty_daily_amended.csv")
display(bank_nifty['date'].dtype)
display(stocks['Date'].dtype)
stocks['Date']=pd.to_datetime(stocks['Date'])
bank_nifty['date']=pd.to_datetime(bank_nifty['date'], format="%d-%m-%Y")

ohlc_stocks_equal= stocks[(stocks['Open'] == stocks['High']) & (stocks['High']==stocks['Low']) & (stocks['Low']==stocks['Close'])]
ohlc_bnf_equal= bank_nifty[(bank_nifty['open'] == bank_nifty['high']) & (bank_nifty['high']==bank_nifty['low']) & (bank_nifty['low']==bank_nifty['close'])]


dtype('O')

dtype('O')

In [394]:
ohlc_stocks_equal

,Date,Ticker,Open,High,Low,Close,Volume
544,2025-03-18,HDFCBANK.NS,830.065613,830.065613,830.065613,830.065613,0
836,2026-05-28,HDFCBANK.NS,746.306519,746.306519,746.306519,746.306519,0
1447,2025-03-18,KOTAKBANK.NS,397.474731,397.474731,397.474731,397.474731,0
1739,2026-05-28,KOTAKBANK.NS,388.030090,388.030090,388.030090,388.030090,0
2350,2025-03-18,SBIN.NS,696.363281,696.363281,696.363281,696.363281,0
2642,2026-05-28,SBIN.NS,967.799988,967.799988,967.799988,967.799988,0
3253,2025-03-18,AUBANK.NS,490.717682,490.717682,490.717682,490.717682,0
3545,2026-05-28,AUBANK.NS,1004.024963,1004.024963,1004.024963,1004.024963,0
4156,2025-03-18,AXISBANK.NS,1032.470703,1032.470703,1032.470703,1032.470703,0
4448,2026-05-28,AXISBANK.NS,1303.094971,1303.094971,1303.094971,1303.094971,0
